<a href="https://colab.research.google.com/github/angamiafatima/CreditScore/blob/main/mxai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installations
!pip install torch torchvision transformers pillow kagglehub scikit-learn matplotlib pandas


MODEL TRAINING

In [ ]:
import os
import json
import glob
import torch
import shutil
import random
from PIL import Image
from torch.utils.data import Dataset, random_split
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig
)
import kagglehub

# 1. CONFIGURATION
class MxAIConfig:
    def __init__(self):
        self.encoder_id = "google/vit-base-patch16-224-in21k"
        self.decoder_id = "gpt2"
        self.max_length = 128
        self.batch_size = 8
        self.epochs = 2
        self.learning_rate = 2e-5
        self.output_dir = "./mxai_checkpoints"
        self.data_fraction = 0.3

conf = MxAIConfig()

# 2. ROBUST DATA PARSING
def download_rico():
    print("Downloading Rico dataset...")
    path = kagglehub.dataset_download("onurgunes1993/rico-dataset")
    print(f"Dataset downloaded to: {path}")
    return path

def parse_rico_json(json_path):
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)

        elements = []
        def traverse(node):
            ui_class = "Element"
            if 'componentLabel' in node:
                ui_class = node['componentLabel']
            elif 'class' in node:
                ui_class = node['class'].split('.')[-1]

            text = node.get('text') or node.get('content-desc') or node.get('label') or ''
            boring_classes = ["View", "ViewGroup", "LinearLayout", "RelativeLayout",
                              "FrameLayout", "DrawerLayout", "CoordinatorLayout"]

            if text:
                elements.append(f"{ui_class}: {text}")
            elif ui_class not in boring_classes:
                elements.append(ui_class)

            if 'children' in node:
                for child in node['children']:
                    traverse(child)

        traverse(data)

        # Cleanup and limit
        unique_elements = list(dict.fromkeys(elements))
        result = " | ".join(unique_elements[:35])

        if len(result) < 5:
            return "UI Screen"
        return result

    except Exception:
        return "UI Screen"

class RicoDataset(Dataset):
    def __init__(self, data_path, processor, tokenizer, max_length=128):
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_length = max_length

        print("Indexing files...")
        all_jpgs = glob.glob(os.path.join(data_path, "**", "*.jpg"), recursive=True)
        all_jsons = glob.glob(os.path.join(data_path, "**", "*.json"), recursive=True)

        json_map = {os.path.splitext(os.path.basename(p))[0]: p for p in all_jsons}

        self.valid_samples = []
        for img_path in all_jpgs:
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            if base_name in json_map:
                self.valid_samples.append((img_path, json_map[base_name]))

        print(f"Found {len(self.valid_samples)} matched Image-JSON pairs.")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        img_path, json_path = self.valid_samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), color='black')

        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        target_text = parse_rico_json(json_path)

        # Tokenize
        model_inputs = self.tokenizer(
            target_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = model_inputs.input_ids.squeeze()
        # Masking the padding tokens so the model doesn't learn them
        # But not masking the EOS token
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values.squeeze(),
            "labels": labels
        }

# 3. VERIFICATION-
def verify_data_integrity(dataset, tokenizer):
    print("\nVERIFYING DATA QUALITY...")
    indices = [0, 10, 20] if len(dataset) > 20 else range(len(dataset))
    for i in indices:
        sample = dataset[i]
        labels = sample['labels'].clone()
        # Temporarily unmask for printing
        labels[labels == -100] = tokenizer.pad_token_id
        text = tokenizer.decode(labels, skip_special_tokens=True)
        print(f"Sample {i}: {text[:100]}...")
    print("Data Verification Complete.\n")

# 4. MAIN EXECUTION
if __name__ == "__main__":
    # A. Download Data
    dataset_path = download_rico()

    # B. Build Model and Tokenizer
    print("Building Model...")

    # 1. Load Tokenizer and Add Pad Token
    tokenizer = AutoTokenizer.from_pretrained(conf.decoder_id)
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

    feature_extractor = ViTImageProcessor.from_pretrained(conf.encoder_id)

    model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
        conf.encoder_id, conf.decoder_id
    )

    # 2. Resize Model to fit new Pad Token
    model.decoder.resize_token_embeddings(len(tokenizer))

    # 3. Configure Model Special Tokens
    model.config.decoder_start_token_id = tokenizer.bos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.vocab_size = len(tokenizer)

    # 4. Configure Inference (Sampling)
    model.generation_config = GenerationConfig(
        decoder_start_token_id=tokenizer.bos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_length=conf.max_length,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2,
        no_repeat_ngram_size=2
    )

    # C. Create Dataset
    full_dataset = RicoDataset(dataset_path, feature_extractor, tokenizer)

    # VALIDATION
    verify_data_integrity(full_dataset, tokenizer)

    # D. Limit Data (30%)
    total_len = len(full_dataset)
    subset_len = int(total_len * conf.data_fraction)
    active_dataset, _ = random_split(full_dataset, [subset_len, total_len - subset_len])

    train_size = int(0.9 * len(active_dataset))
    train_dataset, val_dataset = random_split(active_dataset, [train_size, len(active_dataset) - train_size])

    print(f"Training on {len(train_dataset)} samples.")

    # E. Train
    training_args = Seq2SeqTrainingArguments(
        output_dir=conf.output_dir,
        per_device_train_batch_size=conf.batch_size,
        per_device_eval_batch_size=conf.batch_size,
        predict_with_generate=True,
        eval_strategy="steps",
        eval_steps=500,
        save_steps=1000,
        num_train_epochs=conf.epochs,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        report_to="none"
    )

    # Manually stack the images and labels, the standard collator crashes on images
    def mxai_data_collator(features):
        return {
            'pixel_values': torch.stack([f['pixel_values'] for f in features]),
            'labels': torch.stack([f['labels'] for f in features])
        }

    trainer = Seq2SeqTrainer(
        model=model,
        tokenizer=feature_extractor,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=mxai_data_collator,
    )

    print("Starting Training...")
    trainer.train()

    # F. Save and Zip model
    print("Saving model...")
    model_save_path = "./mxai_final_model"
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    feature_extractor.save_pretrained(model_save_path)

    print("Zipping...")
    shutil.make_archive("mxai_final_model", 'zip', model_save_path)

    print("Attempting download...")
    try:
        from google.colab import files
        files.download("mxai_final_model.zip")
    except:
        print("Download mxai_final_model.zip from Output tab.")

Using Colab cache for faster access to the 'rico-dataset' dataset.
Dataset downloaded to: /kaggle/input/rico-dataset
Building Model...


Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['transformer.h.0.crossattention.c_attn.bias', 'transformer.h.0.crossattention.c_attn.weight', 'transformer.h.0.crossattention.c_proj.bias', 'transformer.h.0.crossattention.c_proj.weight', 'transformer.h.0.crossattention.q_attn.bias', 'transformer.h.0.crossattention.q_attn.weight', 'transformer.h.0.ln_cross_attn.bias', 'transformer.h.0.ln_cross_attn.weight', 'transformer.h.1.crossattention.c_attn.bias', 'transformer.h.1.crossattention.c_attn.weight', 'transformer.h.1.crossattention.c_proj.bias', 'transformer.h.1.crossattention.c_proj.weight', 'transformer.h.1.crossattention.q_attn.bias', 'transformer.h.1.crossattention.q_attn.weight', 'transformer.h.1.ln_cross_attn.bias', 'transformer.h.1.ln_cross_attn.weight', 'transformer.h.10.crossattention.c_attn.bias', 'transformer.h.10.crossattention.c_attn.weight', 'transformer.h.10.crossattention.c_proj.bias', 'transformer.h.10.cros

Indexing files...
✅ Found 66261 matched Image-JSON pairs.

🔎 VERIFYING DATA QUALITY...
Sample 0: PhoneWindow$DecorView | Toolbar | Text: Setup | Icon | Text Button: Yes | Text Button: No | Text: Do...
Sample 10: PhoneWindow$DecorView | Advertisement | Web View...
Sample 20: PhoneWindow$DecorView | Toolbar | Icon | Text: https://ww2.tracfone.com/mobile-chat.html | Image | W...
Data Verification Complete.

Training on 17890 samples.


/tmp/ipython-input-955883396.py:234: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting Training...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,3.946100,3.490725
1000,3.499500,3.336066
1500,3.374400,3.244578
2000,3.301400,3.190328
2500,3.227300,3.154110
3000,3.101300,3.120451
3500,3.085800,3.101228
4000,3.088100,3.085705


Saving model...
Zipping...
Attempting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DATA VISUALISATION


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

def visualize_rico_dataset(dataset_path=None):
    # If user hasn't defined dataset_path yet
    if dataset_path is None:
        print("❗ dataset_path not provided. Please set dataset_path = '/path/to/rico'")
        return

    if not os.path.exists(dataset_path):
        print(f"❗ Path not found: {dataset_path}")
        return

    all_jpgs = glob.glob(os.path.join(dataset_path, "**", "*.jpg"), recursive=True)
    all_jsons = glob.glob(os.path.join(dataset_path, "**", "*.json"), recursive=True)

    image_names = set(os.path.splitext(os.path.basename(p))[0] for p in all_jpgs)
    json_names = set(os.path.splitext(os.path.basename(p))[0] for p in all_jsons)

    matched = image_names.intersection(json_names)
    unmatched_images = image_names - matched
    unmatched_jsons = json_names - matched

    df = pd.DataFrame({
        "Category": ["Matched Pairs", "Unmatched Images", "Unmatched JSON Files"],
        "Count": [len(matched), len(unmatched_images), len(unmatched_jsons)]
    })

    plt.figure(figsize=(8,5))
    plt.bar(df["Category"], df["Count"])
    plt.title("Rico Dataset File Matching Overview")
    plt.ylabel("Count")
    plt.xticks(rotation=15)
    plt.show()

# Call it like this:
# visualize_rico_dataset(dataset_path)


In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt

def visualize_ui_text_length(dataset):
    lengths = []
    for i in tqdm(range(min(2000, len(dataset))), desc="Scanning Descriptions"):
        sample = dataset[i]
        labels = sample["labels"].clone()
        labels[labels == -100] = dataset.tokenizer.pad_token_id
        text = dataset.tokenizer.decode(labels, skip_special_tokens=True)
        lengths.append(len(text.split()))

    plt.figure(figsize=(8,5))
    plt.hist(lengths, bins=40)
    plt.title("Distribution of UI Description Lengths")
    plt.xlabel("Word Count")
    plt.ylabel("Frequency")
    plt.show()

# Example usage after dataset creation:
visualize_ui_text_length(full_dataset)


NameError: name 'full_dataset' is not defined

In [ ]:
import json
import matplotlib.pyplot as plt

def visualize_training_loss(log_path="./mxai_checkpoints/trainer_state.json"):
    with open(log_path, "r") as f:
        logs = json.load(f)

    steps = []
    losses = []

    for entry in logs["log_history"]:
        if "loss" in entry:
            steps.append(entry["step"])
            losses.append(entry["loss"])

    plt.figure(figsize=(8,5))
    plt.plot(steps, losses)
    plt.title("Training Loss Curve")
    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()

visualize_training_loss()


FileNotFoundError: [Errno 2] No such file or directory: './mxai_checkpoints/trainer_state.json'

In [ ]:
def visualize_train_vs_val(log_path="./mxai_checkpoints/trainer_state.json"):
    with open(log_path, "r") as f:
        logs = json.load(f)

    train_steps = []
    train_losses = []
    val_steps = []
    val_losses = []

    for entry in logs["log_history"]:
        if "loss" in entry:
            train_steps.append(entry["step"])
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            val_steps.append(entry["step"])
            val_losses.append(entry["eval_loss"])

    plt.figure(figsize=(8,5))
    plt.plot(train_steps, train_losses, label="Train Loss")
    plt.plot(val_steps, val_losses, label="Validation Loss")
    plt.title("Train vs Validation Loss Curve")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

visualize_train_vs_val()


FileNotFoundError: [Errno 2] No such file or directory: './mxai_checkpoints/trainer_state.json'

In [ ]:
# pip install gradio
import gradio as gr
import torch
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

# ==========================================
# 1. LOAD MODEL ONCE (Global Scope)
# ==========================================
# We load it here so we don't reload it for every single prediction
model_path = "./mxai_final_model"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model on {device}...")
try:
    model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)
    processor = ViTImageProcessor.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure the folder './mxai_final_model' exists and contains the model files.")

# ==========================================
# 2. PREDICTION FUNCTION (With Fixes)
# ==========================================
def predict_ui(image):
    if image is None:
        return "Please upload an image."

    # Process
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    # Generate with Anti-Repetition Settings
    generated_ids = model.generate(
        pixel_values,
        max_length=150,
        num_beams=4,

        # --- THE FIXES FOR "| | |" ---
        no_repeat_ngram_size=2,  # Stops the model from writing the same 2-word phrase twice
        repetition_penalty=1.2,  # Penalizes using the same word repeatedly
        early_stopping=True,     # Stops as soon as the sentence looks finished
        # -----------------------------

        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return text

# 3. LAUNCH UI
interface = gr.Interface(
    fn=predict_ui,
    inputs=gr.Image(type="pil", label="Upload UI Screenshot"),
    outputs="text",
    title="MxAI: UI Understanding Model",
    description="Upload a mobile screen, and I'll tell you what elements are on it."
)

# launch(share=True) creates a public link you can send to friends
interface.launch(share=True)

Loading model on cuda...
Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc40fdff0dc32a37b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import torch
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    AutoTokenizer,
    pipeline
)

# ==========================================
# 1. LOAD "THE EYES" (MxAI Vision Model)
# ==========================================
print("Loading MxAI (Vision)...")
mxai_path = "./mxai_final_model"  # Make sure this folder exists in Colab

try:
    mxai_model = VisionEncoderDecoderModel.from_pretrained(mxai_path)
    mxai_processor = ViTImageProcessor.from_pretrained(mxai_path)
    mxai_tokenizer = AutoTokenizer.from_pretrained(mxai_path)
    print("MxAI Loaded!")
except Exception as e:
    print(f"Error loading MxAI: {e}")

# ==========================================
# 2. LOAD "THE BRAIN" (Flan-T5)
# ==========================================
print("Loading Brain (Flan-T5)...")

llm_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("Brain Loaded!")

# ==========================================
# 3. DEFINE THE PIPELINE
# ==========================================

def get_screen_description(image):
    """Use MxAI Vision model to extract screen elements."""
    if image is None:
        return ""

    pixel_values = mxai_processor(images=image, return_tensors="pt").pixel_values

    generated_ids = mxai_model.generate(
        pixel_values,
        max_length=200,
        do_sample=True,
        top_k=50,
        repetition_penalty=1.2
    )

    description = mxai_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return description


def ask_agent(image, question, history):
    """Combine Vision Output + LLM Reasoning"""

    if image is None:
        return "Please upload an image first."

    print("Scanning screen...")
    screen_content = get_screen_description(image)

    print(f"Raw Vision Data: {screen_content}")  # Debug print

    prompt = f"""
    Context: I am looking at a mobile phone screen.
    Here is the extracted text and UI elements:
    "{screen_content}"

    Question: {question}

    Answer:
    """

    print("Thinking...")
    result = llm_pipeline(prompt, max_length=150, do_sample=False)
    answer = result[0]['generated_text']

    return answer

# ==========================================
# 4. GRADIO UI
# ==========================================
interface = gr.ChatInterface(
    fn=ask_agent,
    additional_inputs=[
        gr.Image(type="pil", label="Upload UI Screenshot")
    ],
    title="MxAI + LLM Brain",
    description="MxAI interprets the screen, and Flan-T5 answers your questions."
)

interface.launch(share=True)


Loading MxAI (Vision)...
Error loading MxAI: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './mxai_final_model'.
Loading Brain (Flan-T5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

KeyboardInterrupt: 